# Kaggle SDXL 最小验证（3 格）

**开跑前**：右上 Settings → Accelerator = `GPU T4 x2`（或 P100）；Internet = `On`；Kaggle 手机验证必须已完成（否则 GPU 不可选）。

跑完把三格输出发回：GPU 型号/显存、每张的加载秒数、推理秒数、尺寸、字节数、md5。


In [ ]:
# 1. 依赖 + 环境
!pip -q install diffusers transformers accelerate safetensors pillow
import subprocess, torch, time
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('vram %.1fGB' % (torch.cuda.get_device_properties(0).total_memory/1024**3))


In [ ]:
# 2. animagine-xl-4.0 fp16 / 832x1216 @28 步
import time, io, hashlib, os
import torch
from diffusers import StableDiffusionXLPipeline
from PIL import Image, ImageStat

t0 = time.time()
pipe = StableDiffusionXLPipeline.from_pretrained('cagliostrolab/animagine-xl-4.0',
                                                 torch_dtype=torch.float16, use_safetensors=True)
pipe = pipe.to('cuda')
load_s = time.time() - t0
print('load %.1fs' % load_s)

t1 = time.time()
img = pipe(prompt='masterpiece, best quality, 1girl, solo, silver long hair, blue eyes, '
                  'frilled maid outfit, sitting on bed, soft window light, detailed skin',
           negative_prompt='worst quality, low quality, blurry, jpeg artifacts, bad anatomy',
           height=1216, width=832, num_inference_steps=28,
           generator=torch.Generator('cuda').manual_seed(4242)).images[0]
infer_s = time.time() - t1
buf = io.BytesIO(); img.save(buf, format='PNG'); blob = buf.getvalue()
open('/kaggle/working/animagine_test.png','wb').write(blob)
print('infer %.1fs | size %s | %dB | md5 %s | mean %.1f'
      % (infer_s, img.size, len(blob), hashlib.md5(blob).hexdigest(),
         ImageStat.Stat(img.convert('L')).mean[0]))
del pipe; torch.cuda.empty_cache()


In [ ]:
# 3. sd-turbo fp16 / 512x768 @4 步
import time, io, hashlib, torch
from diffusers import StableDiffusionPipeline
from PIL import ImageStat

t0 = time.time()
pipe = StableDiffusionPipeline.from_pretrained('stabilityai/sd-turbo',
                                               torch_dtype=torch.float16, use_safetensors=True).to('cuda')
load_s = time.time() - t0
t1 = time.time()
img = pipe(prompt='a red cube on a white table, studio light', height=768, width=512,
           num_inference_steps=4, guidance_scale=0.0,
           generator=torch.Generator('cuda').manual_seed(7)).images[0]
infer_s = time.time() - t1
buf = io.BytesIO(); img.save(buf, format='PNG'); blob = buf.getvalue()
open('/kaggle/working/sdturbo_test.png','wb').write(blob)
print('load %.1fs | infer %.1fs | size %s | %dB | md5 %s | mean %.1f'
      % (load_s, infer_s, img.size, len(blob), hashlib.md5(blob).hexdigest(),
         ImageStat.Stat(img.convert('L')).mean[0]))
